# A3-M1 · 03 transformers 토크나이저 사용법
02에서 저장한 `shop_wordpiece.json`을 transformers 토크나이저로 감싸 사용합니다. 마지막 셀은 Hub의 klue·e5 토크나이저를 내려받습니다.

## 07-1. transformers 토크나이저로 감싸기 — AutoTokenizer로 다시 열기

In [1]:
from transformers import PreTrainedTokenizerFast, AutoTokenizer

hf = PreTrainedTokenizerFast(tokenizer_file="shop_wordpiece.json",
                             unk_token="[UNK]", pad_token="[PAD]", cls_token="[CLS]",
                             sep_token="[SEP]", mask_token="[MASK]", model_max_length=64,
                             model_input_names=["input_ids", "token_type_ids", "attention_mask"])
hf.save_pretrained("shop_tokenizer")                       # 폴더로 저장
tok = AutoTokenizer.from_pretrained("shop_tokenizer")       # Hub 모델과 똑같은 방법으로 열기
print(type(tok).__name__, "| 어휘", tok.vocab_size, "| 최대 길이", tok.model_max_length)
print("특수 토큰:", tok.special_tokens_map)

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TokenizersBackend | 어휘 632 | 최대 길이 64
특수 토큰: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


## 07-2. tok(문장) — 결과는 사전처럼

In [2]:
enc = tok("냉장고 배송비는 얼마인가요?")
print(type(enc).__name__, list(enc.keys()))
print("input_ids:", enc["input_ids"])
print("tokens   :", tok.convert_ids_to_tokens(enc["input_ids"]))
print("tokenize :", tok.tokenize("냉장고 배송비는 얼마인가요?"))
print("decode   :", tok.decode(enc["input_ids"]), "/", tok.decode(enc["input_ids"], skip_special_tokens=True))

BatchEncoding ['input_ids', 'token_type_ids', 'attention_mask']
input_ids: [2, 449, 447, 448, 17, 3]
tokens   : ['[CLS]', '냉장고', '배송비는', '얼마인가요', '?', '[SEP]']
tokenize : ['냉장고', '배송비는', '얼마인가요', '?']
decode   : [CLS] 냉장고 배송비는 얼마인가요? [SEP] / 냉장고 배송비는 얼마인가요?


## 07-3. 여러 문장 — padding·truncation·return_tensors

In [3]:
qs = ["배송 언제 와요?", "냉장고 반품하려면 어떻게 하나요?", "무료 배송 기준이 뭐예요? 3만 원 이상인가요?"]
b = tok(qs, padding=True, truncation=True, max_length=12, return_tensors="np")
for key in b:
    print(key, b[key].shape)
print(b["input_ids"])
print(b["attention_mask"])

print("\npadding='max_length':", tok(qs[0], padding="max_length", max_length=12)["input_ids"])
print("padding_side 왼쪽  :", tok(qs[:2], padding=True, padding_side="left")["input_ids"])

input_ids (3, 12)
token_type_ids (3, 12)
attention_mask (3, 12)
[[  2 341 346   1  17   3   0   0   0   0   0   0]
 [  2 449 436 432 427  17   3   0   0   0   0   0]
 [  2 367 341 532 217   1  17 535 134 537 352   3]]
[[1 1 1 1 1 1 0 0 0 0 0 0]
 [1 1 1 1 1 1 1 0 0 0 0 0]
 [1 1 1 1 1 1 1 1 1 1 1 1]]

padding='max_length': [2, 341, 346, 1, 17, 3, 0, 0, 0, 0, 0, 0]
padding_side 왼쪽  : [[0, 2, 341, 346, 1, 17, 3], [2, 449, 436, 432, 427, 17, 3]]


## 08-1. 문장 쌍 — 질문과 문서를 한 입력으로

In [4]:
p = tok("배송비 얼마예요?", "3만 원 이상 구매 시 무료 배송입니다.")
print(tok.convert_ids_to_tokens(p["input_ids"]))
print("token_type_ids:", p["token_type_ids"])
long_q = "배송비 " * 10
p2 = tok(long_q, "3만 원 이상 무료 배송", truncation="only_first", max_length=12)
print("질문만 자르기:", tok.convert_ids_to_tokens(p2["input_ids"]))

['[CLS]', '배송', '##비', '[UNK]', '?', '[SEP]', '3만', '원', '이상', '구매', '시', '무료', '배송입니다', '.', '[SEP]']
token_type_ids: [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
질문만 자르기: ['[CLS]', '배송', '##비', '배송', '##비', '[SEP]', '3만', '원', '이상', '무료', '배송', '[SEP]']


## 08-2. offsets_mapping · word_ids — 토큰을 원문 위치로

In [5]:
text = "냉장고 배송비는 얼마인가요?"
o = tok(text, return_offsets_mapping=True)
for tid, (s, e), w in zip(o["input_ids"], o["offset_mapping"], o.word_ids()):
    print(f"{tok.convert_ids_to_tokens(tid):<8} 위치 {s:>2}~{e:<2} 원문 '{text[s:e]}'  단어번호 {w}")

[CLS]    위치  0~0  원문 ''  단어번호 None
냉장고      위치  0~3  원문 '냉장고'  단어번호 0
배송비는     위치  4~8  원문 '배송비는'  단어번호 1
얼마인가요    위치  9~14 원문 '얼마인가요'  단어번호 2
?        위치 14~15 원문 '?'  단어번호 3
[SEP]    위치  0~0  원문 ''  단어번호 None


## 08-3. 긴 문서 — 잘라 버리지 말고 겹쳐서 나누기(stride)

In [6]:
doc = ("냉장고 같은 대형 가전은 설치 기사가 직접 방문합니다. 방문 일정은 문자로 안내됩니다. "
       "제주와 도서산간은 3~5일 더 걸립니다. 설치 후 포장재는 기사님이 수거합니다.")
print("전체 토큰 수:", len(tok(doc)["input_ids"]))

cut = tok(doc, truncation=True, max_length=16)
print("truncation만:", tok.decode(cut["input_ids"], skip_special_tokens=True), "← 뒤는 사라짐")

ch = tok(doc, truncation=True, max_length=16, stride=4,
         return_overflowing_tokens=True, return_offsets_mapping=True)
print("조각 수:", len(ch["input_ids"]))
for i, (ids, offs) in enumerate(zip(ch["input_ids"], ch["offset_mapping"])):
    real = [o for o in offs if o != (0, 0)]                     # 특수 토큰 제외
    print(f"조각 {i}: 원문 {real[0][0]}~{real[-1][1]} → {doc[real[0][0]:real[-1][1]]}")

전체 토큰 수: 36
truncation만: 냉장고 대형 가전은 설치 기사가 직접 방문합니다. 방문 일정은 문자 ← 뒤는 사라짐
조각 수: 3
조각 0: 원문 0~40 → 냉장고 같은 대형 가전은 설치 기사가 직접 방문합니다. 방문 일정은 문자
조각 1: 원문 34~63 → 일정은 문자로 안내됩니다. 제주와 도서산간은 3~5일
조각 2: 원문 53~93 → 도서산간은 3~5일 더 걸립니다. 설치 후 포장재는 기사님이 수거합니다.


## 08-4. add_tokens — 도메인 단어를 한 덩어리로

In [7]:
w = "RF85T 머시닝센터"
print("추가 전:", tok.tokenize(w))
n = tok.add_tokens(["RF85T", "머시닝센터"])
print("추가한 수:", n, "| 어휘", len(tok))
print("추가 후:", tok.tokenize(w))
print("주의: 모델과 함께 쓸 때는 model.resize_token_embeddings(len(tok)) 필요")

추가 전: ['rf85t', '[UNK]']
추가한 수: 2 | 어휘 634
추가 후: ['rf85t', '머시닝센터']
주의: 모델과 함께 쓸 때는 model.resize_token_embeddings(len(tok)) 필요


## 09-1. 사전학습 토크나이저 — klue/bert-base와 e5 (인터넷 필요)

In [8]:
from transformers import AutoTokenizer
for name in ["klue/bert-base", "intfloat/multilingual-e5-small"]:
    t = AutoTokenizer.from_pretrained(name)
    s = "냉장고 배송비는 얼마인가요? RF85T"
    print(name, "|", type(t).__name__, "| 어휘", t.vocab_size, "| 최대", t.model_max_length)
    print("   ", t.tokenize(s))
    print("   ", t(s)["input_ids"])

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--klue--bert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


klue/bert-base | BertTokenizer | 어휘 32000 | 최대 512
    ['냉장고', '배송', '##비', '##는', '얼마', '##인', '##가요', '?', 'RF', '##85', '##T']
    [2, 8545, 9488, 2151, 2259, 3943, 2179, 18119, 35, 26048, 21411, 2081, 3]
intfloat/multilingual-e5-small | XLMRobertaTokenizer | 어휘 250002 | 최대 512
    ['▁냉', '장', '고', '▁배송', '비', '는', '▁얼마', '인', '가요', '?', '▁RF', '85', 'T']
    [0, 129673, 3013, 1077, 112667, 4274, 769, 131602, 1571, 82484, 32, 100250, 12951, 618, 2]
